# C1 data cleaning

Track C, Apetit Climate & Nature Navigator 2026.

This notebook fixes errors in the farm yield history and builds the table the yield model trains on. Each row is one farm, one crop and one year, with soil and weather-cell information added.

The original files are never changed. Every fix is written to `outputs/corrections_log.csv` with the record, the old and new value, the rule that found it, the evidence, and who proposed and approved it. The cleaned tables are the original data plus the approved fixes, so any number can be traced back to its source row.

| Rule | Problem | What happens |
|---|---|---|
| R1 | Yield typed 10 times too large | Divided by 10 |
| R2 | Nitrogen rate with the decimal point in the wrong place | Multiplied or divided by 10 |
| R3 | Survey area doesn't match the farm's parcels | Replaced with the parcel total |
| R4 | Organic soil reported as mineral | Flagged for a person to check, not changed |
| R5 | Farm latitude and longitude swapped | Swapped back |

A rule's fixes are approved automatically only if, on the public 2023–24 answer key, it flagged no correct records and its fixed values match the true ones (section 4). Everything else waits for a person. A reviewer can approve or reject any single fix in `review/decisions.csv`, and the next run uses their decision.

To run it, open the notebook in this folder with the venv and choose Run All. If the challenge package isn't at `../APETIT_CLIMATE_NATURE_NAVIGATOR_2026`, set `CCN_PKG` to its path.

## 0. Setup

In [1]:
from pathlib import Path
import os, json, hashlib, datetime as dt
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

PKG = Path(os.environ.get("CCN_PKG", "../APETIT_CLIMATE_NATURE_NAVIGATOR_2026")).resolve()
DATA = PKG / "data"
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
REVIEW = Path("review"); REVIEW.mkdir(exist_ok=True)
RUN_TS = dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()
assert (DATA / "03_farming" / "farm_survey_fi.csv").exists(), f"Package not found at {PKG}"

# Rule parameters (all in one place so the method note can quote them)
PARAMS = {
    "R1_yield_rel_to_crop_year_median_max": 5.0,   # real yields stay below ~1.5x the median
    "R2_n_ratio_high": 5.0,                         # shifted values sit at ~10x / ~0.1x
    "R2_n_ratio_low": 0.2,
    "R3_area_tolerance": 0.01,                # rounding noise is < 0.3 %
    "R4_organic_share_threshold": 0.30,             # A1 definition of soil_misreport
    "contract_crops": ["pea", "turnip_rape", "potato", "carrot", "spinach"],
}
print("package:", PKG.name)
print("run:", RUN_TS)

package: APETIT_CLIMATE_NATURE_NAVIGATOR_2026
run: 2026-11-06T19:37:03+00:00


## 1. Load the raw files

The files are read exactly as delivered. The Foodhills file uses semicolons and decimal commas. A checksum of each file goes into `run_metadata.json`, so you can tell later which version of the data a run used.

In [2]:
FILES = {
    "farm_survey":   ("03_farming/farm_survey_fi.csv", {}),
    "foodhills":     ("03_farming/foodhills_odlingsdata_se.csv", {"sep": ";", "decimal": ","}),
    "farm_registry": ("01_master/farm_registry.csv", {}),
    "parcels":       ("01_master/parcels.csv", {}),
    "crop_history":  ("03_farming/parcel_crop_history_and_plan.csv", {}),
    "grid":          ("01_master/climate_grid_cells.csv", {}),
}
raw, checksums = {}, {}
for name, (rel, kw) in FILES.items():
    path = DATA / rel
    raw[name] = pd.read_csv(path, **kw)
    checksums[f"data/{rel}"] = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{name:14s} {raw[name].shape}")

labels = pd.read_csv(PKG / "tasks/dev/A1_dev_issue_labels_2023_2024.csv")
template_c1 = pd.read_csv(PKG / "tasks/templates/C1_yields_2027_2031_submission.csv")

farm_survey    (3437, 16)
foodhills      (2683, 8)
farm_registry  (680, 10)
parcels        (14213, 13)
crop_history   (284260, 6)
grid           (84, 6)


### Helper: declared area

For each farm, crop and year, this adds up the area of the parcels the farm declared for that crop. Several rules compare the survey against it.

In [3]:
decl = (raw["crop_history"].query("status == 'declared'")
        .merge(raw["parcels"], on=["parcel_id", "farm_id"], how="left", validate="many_to_one"))
decl["org_area"] = np.where(decl["soil_class_soilmap"].eq("organic"), decl["area_ha"], 0.0)
decl_fcy = (decl.groupby(["farm_id", "year", "crop"])
            .agg(decl_area_ha=("area_ha", "sum"), org_area=("org_area", "sum")).reset_index())
decl_fcy["organic_share_soilmap"] = decl_fcy["org_area"] / decl_fcy["decl_area_ha"]

LOG_COLS = ["correction_id", "rule_id", "issue_type", "table", "record_key", "field",
            "old_value", "new_value", "action", "evidence", "proposed_by", "proposed_at"]

def proposals(rule_id, issue_type, table, record_key, field, old, new, evidence, action):
    d = pd.DataFrame({
        "rule_id": rule_id, "issue_type": issue_type, "table": table,
        "record_key": list(record_key), "field": field,
        "old_value": list(old), "new_value": list(new),
        "action": action, "evidence": list(evidence),
    })
    d["correction_id"] = d["rule_id"] + "|" + d["record_key"] + "|" + d["field"]  # stable across runs
    d["proposed_by"] = f"c1_clean.ipynb/{rule_id}"
    d["proposed_at"] = RUN_TS
    return d[LOG_COLS]

fs = raw["farm_survey"].copy()
fs["record_key"] = "farm_survey:" + fs["survey_id"]
fs = fs.merge(decl_fcy, on=["farm_id", "year", "crop"], how="left", validate="one_to_one")
assert fs["decl_area_ha"].notna().all(), "survey rows without declared parcels"

## 2. Find problems

Each rule below lists the fixes it would make. Nothing is changed yet.

### R1: yield typed 10 times too large

Each yield is compared with the median for the same crop and year. Real yields, bad years included, stay below about 1.5 times the median. Typing errors sit near 10 times. Low yields are left alone: poor seasons are real, and C2 needs them to measure how often bad years happen.

In [4]:
med = fs.groupby(["crop", "year"])["yield_t_ha"].transform("median")
rel = fs["yield_t_ha"] / med
h = fs[rel > PARAMS["R1_yield_rel_to_crop_year_median_max"]]
r1 = proposals("R1", "outlier_implausible", "farm_survey", h["record_key"], "yield_t_ha",
               h["yield_t_ha"], (h["yield_t_ha"] / 10).round(3),
               [f"{y} t/ha is {r:.1f}x the {c} {yr} median ({m:.2f}); /10 = {y/10:.2f}"
                for y, r, c, yr, m in zip(h["yield_t_ha"], rel[h.index], h["crop"], h["year"], med[h.index])],
               "fix")
r1

,correction_id,rule_id,issue_type,table,record_key,field,old_value,new_value,action,evidence,proposed_by,proposed_at
0,R1|farm_survey:FS00236|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS00236,yield_t_ha,288.8,28.88,fix,288.8 t/ha is 8.7x the potato 2013 median (33.27); /10 = 28.88,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
1,R1|farm_survey:FS00237|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS00237,yield_t_ha,369.5,36.95,fix,369.5 t/ha is 11.5x the potato 2016 median (32.05); /10 = 36.95,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
2,R1|farm_survey:FS00388|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS00388,yield_t_ha,43.1,4.31,fix,43.1 t/ha is 13.7x the pea 2019 median (3.15); /10 = 4.31,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
3,R1|farm_survey:FS00604|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS00604,yield_t_ha,31.1,3.11,fix,31.1 t/ha is 8.2x the pea 2017 median (3.77); /10 = 3.11,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
4,R1|farm_survey:FS00778|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS00778,yield_t_ha,286.9,28.69,fix,286.9 t/ha is 8.6x the potato 2013 median (33.27); /10 = 28.69,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
5,R1|farm_survey:FS00919|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS00919,yield_t_ha,27.3,2.73,fix,27.3 t/ha is 10.0x the pea 2021 median (2.74); /10 = 2.73,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
6,R1|farm_survey:FS00932|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS00932,yield_t_ha,31.9,3.19,fix,31.9 t/ha is 9.6x the pea 2025 median (3.33); /10 = 3.19,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
7,R1|farm_survey:FS01267|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS01267,yield_t_ha,21.9,2.19,fix,21.9 t/ha is 8.1x the pea 2023 median (2.72); /10 = 2.19,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
8,R1|farm_survey:FS01375|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS01375,yield_t_ha,30.0,3.00,fix,30.0 t/ha is 10.9x the pea 2021 median (2.74); /10 = 3.00,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00
9,R1|farm_survey:FS01726|yield_t_ha,R1,outlier_implausible,farm_survey,farm_survey:FS01726,yield_t_ha,14.0,1.40,fix,14.0 t/ha is 8.5x the turnip_rape 2020 median (1.65); /10 = 1.40,c1_clean.ipynb/R1,2026-11-06T19:37:03+00:00


### R2: nitrogen rate with a misplaced decimal point

Each rate is compared with the same farm's usual rate for that crop, or with the crop's usual rate if the farm normally applies none. A rate of zero is normal for some crops, such as peas, and is skipped.

In [5]:
n = fs["n_mineral_kg_ha"]
farm_med = fs.groupby(["farm_id", "crop"])["n_mineral_kg_ha"].transform("median")
crop_med = fs.groupby("crop")["n_mineral_kg_ha"].transform("median")
ref = farm_med.where(farm_med > 0, crop_med)
ratio_n = n / ref
hi = ratio_n > PARAMS["R2_n_ratio_high"]
lo = (ratio_n < PARAMS["R2_n_ratio_low"]) & (n > 0)
h = fs[hi | lo]
new = np.where(hi[h.index], h["n_mineral_kg_ha"] / 10, h["n_mineral_kg_ha"] * 10).round(1)
r2 = proposals("R2", "decimal_shift", "farm_survey", h["record_key"], "n_mineral_kg_ha",
               h["n_mineral_kg_ha"], new,
               [f"{v} kg N/ha is {r:.2f}x the farm/crop reference ({f:.1f}); shifted value {nv}"
                for v, r, f, nv in zip(h["n_mineral_kg_ha"], ratio_n[h.index], ref[h.index], new)],
               "fix")
print(len(r2), "proposals"); r2.head()

44 proposals


,correction_id,rule_id,issue_type,table,record_key,field,old_value,new_value,action,evidence,proposed_by,proposed_at
0,R2|farm_survey:FS00052|n_mineral_kg_ha,R2,decimal_shift,farm_survey,farm_survey:FS00052,n_mineral_kg_ha,1503.0,150.3,fix,1503.0 kg N/ha is 11.07x the farm/crop reference (135.8); shifted value 150.3,c1_clean.ipynb/R2,2026-11-06T19:37:03+00:00
1,R2|farm_survey:FS00064|n_mineral_kg_ha,R2,decimal_shift,farm_survey,farm_survey:FS00064,n_mineral_kg_ha,1.9,19.0,fix,1.9 kg N/ha is 0.10x the farm/crop reference (19.0); shifted value 19.0,c1_clean.ipynb/R2,2026-11-06T19:37:03+00:00
2,R2|farm_survey:FS00110|n_mineral_kg_ha,R2,decimal_shift,farm_survey,farm_survey:FS00110,n_mineral_kg_ha,2.3,23.0,fix,2.3 kg N/ha is 0.09x the farm/crop reference (26.1); shifted value 23.0,c1_clean.ipynb/R2,2026-11-06T19:37:03+00:00
3,R2|farm_survey:FS00136|n_mineral_kg_ha,R2,decimal_shift,farm_survey,farm_survey:FS00136,n_mineral_kg_ha,2.5,25.0,fix,2.5 kg N/ha is 0.10x the farm/crop reference (24.1); shifted value 25.0,c1_clean.ipynb/R2,2026-11-06T19:37:03+00:00
4,R2|farm_survey:FS00142|n_mineral_kg_ha,R2,decimal_shift,farm_survey,farm_survey:FS00142,n_mineral_kg_ha,2.2,22.0,fix,2.2 kg N/ha is 0.10x the farm/crop reference (22.9); shifted value 22.0,c1_clean.ipynb/R2,2026-11-06T19:37:03+00:00


### R3: survey area doesn't match the parcels

The area in the survey should equal the total area of the parcels the farm declared for that crop and year. Normal rounding differences are under 0.3%. The planted errors are 0.6, 1.35 or 1.6 times the true area.

In [6]:
ratio_a = fs["area_ha"] / fs["decl_area_ha"]
h = fs[(ratio_a - 1).abs() > PARAMS["R3_area_tolerance"]]
r3 = proposals("R3", "area_mismatch", "farm_survey", h["record_key"], "area_ha",
               h["area_ha"], h["decl_area_ha"].round(2),
               [f"survey {a} ha vs {d:.2f} ha in declared parcels (ratio {a/d:.2f})"
                for a, d in zip(h["area_ha"], h["decl_area_ha"])],
               "fix")
print(len(r3), "proposals"); print(ratio_a[(ratio_a - 1).abs() > 0.01].round(2).value_counts())

85 proposals
1.60    32
0.60    28
1.35    25
Name: count, dtype: int64


### R4: organic soil reported as mineral

A record is flagged when the soil map puts at least 30% of the crop's area on organic soil but the farmer reported mineral soil. The soil map is only about 88% accurate, so these records go to a person instead of being changed. The model ignores the farmer's soil answer anyway and uses the soil-map share (`organic_share_soilmap`).

In [7]:
h = fs[(fs["organic_share_soilmap"] >= PARAMS["R4_organic_share_threshold"])
       & fs["soil_type_reported"].eq("Mineral soil")]
r4 = proposals("R4", "soil_misreport", "farm_survey", h["record_key"], "soil_type_reported",
               h["soil_type_reported"], [f"organic share {s:.2f} (soil map)" for s in h["organic_share_soilmap"]],
               [f"{o:.1f} of {d:.1f} ha on organic soil per soil map (~88% accurate)"
                for o, d in zip(h["org_area"], h["decl_area_ha"])],
               "flag")
print(len(r4), "flags")

195 flags


### R5: latitude and longitude swapped

All the farms lie between 55 and 62 °N and between 12 and 24 °E, so a latitude smaller than the longitude means the two were swapped. The farm's parcel locations confirm each case.

In [8]:
fr = raw["farm_registry"].copy()
cent = raw["parcels"].groupby("farm_id")[["lat", "lon"]].median()
h = fr[fr["lat"] < fr["lon"]].join(cent, on="farm_id", rsuffix="_parcels")
ev = [f"lat {a} < lon {b}; parcel centroid at ({c:.3f}, {d:.3f})"
      for a, b, c, d in zip(h["lat"], h["lon"], h["lat_parcels"], h["lon_parcels"])]
keys = "farm:" + h["farm_id"]
r5 = pd.concat([
    proposals("R5", "coordinate_swap", "farm_registry", keys, "lat", h["lat"], h["lon"], ev, "fix"),
    proposals("R5", "coordinate_swap", "farm_registry", keys, "lon", h["lon"], h["lat"], ev, "fix"),
])
r5

,correction_id,rule_id,issue_type,table,record_key,field,old_value,new_value,action,evidence,proposed_by,proposed_at
0,R5|farm:F0028|lat,R5,coordinate_swap,farm_registry,farm:F0028,lat,21.94130,61.11341,fix,"lat 21.9413 < lon 61.11341; parcel centroid at (61.115, 21.937)",c1_clean.ipynb/R5,2026-11-06T19:37:03+00:00
1,R5|farm:F0180|lat,R5,coordinate_swap,farm_registry,farm:F0180,lat,23.16414,60.89255,fix,"lat 23.16414 < lon 60.89255; parcel centroid at (60.889, 23.164)",c1_clean.ipynb/R5,2026-11-06T19:37:03+00:00
2,R5|farm:F0223|lat,R5,coordinate_swap,farm_registry,farm:F0223,lat,23.62987,60.74543,fix,"lat 23.62987 < lon 60.74543; parcel centroid at (60.748, 23.621)",c1_clean.ipynb/R5,2026-11-06T19:37:03+00:00
0,R5|farm:F0028|lon,R5,coordinate_swap,farm_registry,farm:F0028,lon,61.11341,21.94130,fix,"lat 21.9413 < lon 61.11341; parcel centroid at (61.115, 21.937)",c1_clean.ipynb/R5,2026-11-06T19:37:03+00:00
1,R5|farm:F0180|lon,R5,coordinate_swap,farm_registry,farm:F0180,lon,60.89255,23.16414,fix,"lat 23.16414 < lon 60.89255; parcel centroid at (60.889, 23.164)",c1_clean.ipynb/R5,2026-11-06T19:37:03+00:00
2,R5|farm:F0223|lon,R5,coordinate_swap,farm_registry,farm:F0223,lon,60.74543,23.62987,fix,"lat 23.62987 < lon 60.74543; parcel centroid at (60.748, 23.621)",c1_clean.ipynb/R5,2026-11-06T19:37:03+00:00


In [9]:
proposed = pd.concat([r1, r2, r3, r4, r5], ignore_index=True)
proposed.groupby(["rule_id", "issue_type", "action"]).size().rename("n").to_frame()

,,,n
rule_id,issue_type,action,
R1,outlier_implausible,fix,18
R2,decimal_shift,fix,44
R3,area_mismatch,fix,85
R4,soil_misreport,flag,195
R5,coordinate_swap,fix,6


## 3. Foodhills (Sweden)

The Swedish file needs translating but no fixes. The columns get English names, and yield is worked out as delivered tonnes divided by area. The checks confirm there are no duplicate rows and that each area matches the declared parcels.

In [10]:
se = raw["foodhills"].rename(columns={
    "odlar_id": "farm_id", "skördeår": "year", "gröda": "crop_sv", "areal_ha": "area_ha",
    "levererad_mängd_ton": "delivered_t", "kvävegiva_kg_ha": "n_mineral_kg_ha",
    "diesel_l_ha": "diesel_l_ha", "bevattnad": "irrigated_sv"})
assert set(se["crop_sv"]) == {"Ärt (frysärt)"} and set(se["irrigated_sv"]) <= {"ja", "nej"}
se["crop"] = "pea"
se["irrigated"] = se["irrigated_sv"].eq("ja")
se["yield_t_ha"] = se["delivered_t"] / se["area_ha"]
se["record_key"] = "foodhills:" + se["farm_id"] + ":" + se["year"].astype(str)
se = se.merge(decl_fcy, on=["farm_id", "year", "crop"], how="left", validate="one_to_one")

assert not se.duplicated(["farm_id", "year"]).any()
assert ((se["area_ha"] / se["decl_area_ha"] - 1).abs() < PARAMS["R3_area_tolerance"]).all()
print(se["yield_t_ha"].describe().round(2).to_dict())

{'count': 2683.0, 'mean': 3.35, 'std': 0.72, 'min': 1.42, '25%': 2.86, '50%': 3.34, '75%': 3.83, 'max': 5.79}


## 4. Check the rules against the 2023–24 answer key

The challenge package includes the true errors for 2023 and 2024, with the correct values. A rule's fixes are approved automatically only if it flagged no correct records there and its fixed values match the true ones. R5 has no entries in the answer key. Its fixes are approved because a swap is unambiguous and the parcel locations confirm it.

In [11]:
lab = labels[labels["table"].eq("farm_survey")].copy()
lab["record_key"] = lab["record_key"].astype(str)
year_of = fs.set_index("record_key")["year"]

rows = []
for rule_id, grp in proposed[proposed["table"].eq("farm_survey")].groupby("rule_id"):
    itype = grp["issue_type"].iat[0]
    dev = grp[grp["record_key"].map(year_of).isin([2023, 2024])]
    truth = lab[lab["issue_type"].eq(itype)]
    tp_keys = set(dev["record_key"]) & set(truth["record_key"])
    tp, fp, fn = len(tp_keys), len(set(dev["record_key"]) - tp_keys), len(set(truth["record_key"]) - tp_keys)
    value_ok = np.nan
    if grp["action"].iat[0] == "fix" and tp:
        m = dev[dev["record_key"].isin(tp_keys)].merge(truth[["record_key", "true_value"]], on="record_key")
        value_ok = np.mean(np.isclose(m["new_value"].astype(float), m["true_value"].astype(float), rtol=0.01, atol=0.11))
    rows.append({"rule_id": rule_id, "issue_type": itype, "dev_labels": len(truth), "dev_flagged": len(dev),
                 "tp": tp, "fp": fp, "fn": fn,
                 "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1),
                 "fixed_value_matches_truth": value_ok, "flagged_all_years": len(grp)})
validation = pd.DataFrame(rows)
validation.loc[len(validation)] = {"rule_id": "R5", "issue_type": "coordinate_swap", "flagged_all_years": len(r5) // 2}
validation.round(3)

,rule_id,issue_type,dev_labels,dev_flagged,tp,fp,fn,precision,recall,fixed_value_matches_truth,flagged_all_years
0,R1,outlier_implausible,2.0,2.0,2.0,0.0,0.0,1.000,1.000,1.0,18
1,R2,decimal_shift,6.0,6.0,6.0,0.0,0.0,1.000,1.000,1.0,44
2,R3,area_mismatch,18.0,18.0,18.0,0.0,0.0,1.000,1.000,1.0,85
3,R4,soil_misreport,33.0,30.0,23.0,7.0,10.0,0.767,0.697,NaN,195
4,R5,coordinate_swap,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3


## 5. Review decisions

By default, fixes from rules that passed section 4 are approved. Other fixes stay `pending`, and R4 records stay `flag_only`. Any decisions in `review/decisions.csv` are then applied on top, matched by `correction_id`. Only approved fixes reach the cleaned tables.

In [12]:
AUTO_APPROVE = {}
for _, v in validation.iterrows():
    if v["rule_id"] == "R5":
        AUTO_APPROVE["R5"] = "rule policy: deterministic swap confirmed by parcel centroids"
    elif v["precision"] == 1.0 and v["fixed_value_matches_truth"] == 1.0:
        AUTO_APPROVE[v["rule_id"]] = (f"rule policy: precision 1.0 ({int(v['tp'])}/{int(v['dev_flagged'])}) "
                                     "and values match truth on 2023-24 labels")
print(json.dumps(AUTO_APPROVE, indent=1))

log = proposed.copy()
log["status"] = np.where(log["action"].eq("flag"), "flag_only",
                np.where(log["rule_id"].isin(list(AUTO_APPROVE)), "approved", "pending"))
log["approved_by"] = np.where(log["status"].eq("approved"), log["rule_id"].map(AUTO_APPROVE), "")
log["review_comment"] = ""

DECISIONS = REVIEW / "decisions.csv"
if not DECISIONS.exists():
    pd.DataFrame(columns=["correction_id", "decision", "reviewer", "comment", "decided_at"]).to_csv(DECISIONS, index=False)
dec = pd.read_csv(DECISIONS, dtype=str).dropna(subset=["correction_id", "decision"])
unknown = set(dec["correction_id"]) - set(log["correction_id"])
assert not unknown, f"decisions for unknown corrections: {sorted(unknown)[:5]}"
for _, d in dec.iterrows():
    i = log["correction_id"].eq(d["correction_id"])
    log.loc[i, "status"] = {"approve": "approved", "reject": "rejected"}[d["decision"].strip().lower()]
    log.loc[i, "approved_by"] = f"{d['reviewer']} ({d.get('decided_at', '')})"
    log.loc[i, "review_comment"] = d.get("comment", "")
print(f"{len(dec)} human decisions applied")
log.groupby(["rule_id", "status"]).size().unstack(fill_value=0)

{
 "R1": "rule policy: precision 1.0 (2/2) and values match truth on 2023-24 labels",
 "R2": "rule policy: precision 1.0 (6/6) and values match truth on 2023-24 labels",
 "R3": "rule policy: precision 1.0 (18/18) and values match truth on 2023-24 labels",
 "R5": "rule policy: deterministic swap confirmed by parcel centroids"
}
0 human decisions applied


status,approved,flag_only
rule_id,,
R1,18,0
R2,44,0
R3,85,0
R4,0,195
R5,6,0


## 6. Apply the approved fixes

In [13]:
def apply_corrections(df, table):
    df = df.copy()
    fixes = log[log["table"].eq(table) & log["status"].eq("approved") & log["action"].eq("fix")]
    for field, grp in fixes.groupby("field"):
        new = grp.set_index("record_key")["new_value"].astype(float)
        i = df["record_key"].isin(new.index)
        df.loc[i, field] = df.loc[i, "record_key"].map(new)
    n_fix = fixes.groupby("record_key").size()
    df["n_corrections"] = df["record_key"].map(n_fix).fillna(0).astype(int)
    return df

fs_clean = apply_corrections(fs, "farm_survey")
fs_clean["soil_misreport_flag"] = fs_clean["record_key"].isin(log.loc[log["rule_id"].eq("R4"), "record_key"])
fr["record_key"] = "farm:" + fr["farm_id"]
fr_clean = apply_corrections(fr, "farm_registry")

before = fs.groupby("crop")["yield_t_ha"].agg(["mean", "max"]).add_suffix("_raw")
after = fs_clean.groupby("crop")["yield_t_ha"].agg(["mean", "max"]).add_suffix("_clean")
before.join(after).round(2)

,mean_raw,max_raw,mean_clean,max_clean
crop,,,,
carrot,50.31,67.37,50.31,67.37
pea,3.44,43.10,3.28,5.67
potato,35.75,369.50,29.49,40.45
spinach,22.73,29.27,22.73,29.27
turnip_rape,1.52,21.50,1.46,2.63


## 7. Match each parcel to a weather cell

Parcels have coordinates but no weather cell. Each parcel gets the nearest of the 84 cells in its region. The cells are a quarter of a degree apart, so no parcel should be more than about 15 km from its cell.

In [14]:
P = raw["parcels"].merge(fr_clean[["farm_id", "region", "country"]], on="farm_id", how="left", validate="many_to_one")
assert P["region"].notna().all()
grid = raw["grid"]

def nearest_cell(points, cells):
    out = []
    for region, sub in points.groupby("region"):
        g = cells[cells["region"].eq(region)]
        la1, lo1 = np.radians(sub["lat"].values)[:, None], np.radians(sub["lon"].values)[:, None]
        la2, lo2 = np.radians(g["lat"].values)[None, :], np.radians(g["lon"].values)[None, :]
        a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
        d = 2 * 6371 * np.arcsin(np.sqrt(a))
        j = d.argmin(axis=1)
        out.append(pd.DataFrame({"parcel_id": sub["parcel_id"].values, "cell_id": g["cell_id"].values[j],
                                 "cell_dist_km": d[np.arange(len(sub)), j].round(2)}))
    return pd.concat(out, ignore_index=True)

cellmap = nearest_cell(P, grid)
P = P.merge(cellmap, on="parcel_id", validate="one_to_one")
assert P["cell_dist_km"].max() < 20, P["cell_dist_km"].max()

# Sanity: after R5 every farm sits within 10 km of its own parcels
far = fr_clean.join(cent, on="farm_id", rsuffix="_p")
far_km = 111 * np.hypot(far["lat"] - far["lat_p"], (far["lon"] - far["lon_p"]) * np.cos(np.radians(far["lat"])))
assert far_km.max() < 10, far.loc[far_km.idxmax()]
print(P.groupby("region")["cell_id"].nunique().to_dict(), "cells used;",
      "max parcel-cell distance", P["cell_dist_km"].max(), "km")

{'FI_SW': 35, 'SE_SKANE': 35} cells used; max parcel-cell distance 15.82 km


## 8. Parcel-year table

One row for each parcel and year with a contract crop, from 2012 to 2031. Years up to 2026 are what farmers declared, and 2027–31 is the crop plan. Each row has the parcel's soil, weather cell and previous crop. The model learns from farm totals built from this table, and predicts C1 parcel by parcel from it.

In [15]:
ch = raw["crop_history"].sort_values(["parcel_id", "year"]).copy()
ch["prev_crop"] = ch.groupby("parcel_id")["crop"].shift(1)
ch.loc[ch.groupby("parcel_id")["year"].diff().ne(1), "prev_crop"] = np.nan   # only true previous year

py = (ch[ch["crop"].isin(PARAMS["contract_crops"])]
      .merge(P.drop(columns=["lat", "lon"]).rename(columns={"area_ha": "parcel_area_ha"}),
             on=["parcel_id", "farm_id"], how="left", validate="many_to_one"))
assert py["cell_id"].notna().all()

# every C1 template row must be in the frame, with the same area
chk = template_c1.merge(py, on=["parcel_id", "farm_id", "year", "crop"], how="left")
assert chk["cell_id"].notna().all(), "template rows missing from parcel-year frame"
assert np.allclose(chk["area_ha"], chk["parcel_area_ha"], atol=0.01)
print(py.shape, py.groupby("status").size().to_dict())

(48401, 20) {'declared': 36337, 'planned': 12064}


## 9. Training table

The cleaned Finnish survey and the Swedish Foodhills rows, with parcel information averaged over each farm's declared parcels and weighted by area. `main_cell_id` is the weather cell that holds most of the farm's area for that crop. Weather can also be averaged across all of a farm's cells using the parcel-year table.

In [16]:
d = py[py["status"].eq("declared")].copy()
A = d["parcel_area_ha"]
d["clay_x_a"] = d["clay_pct_soilmap"] * A
d["slope_x_a"] = d["slope_pct"] * A
d["org_a"] = np.where(d["soil_class_soilmap"].eq("organic"), A, 0.0)
d["sandy_a"] = np.where(d["soil_class_soilmap"].isin(["sand", "fine_sand"]), A, 0.0)
d["same_prev_a"] = np.where(d["prev_crop"].eq(d["crop"]), A, 0.0)
d["catch_a"] = np.where(d["catch_crop"].astype(bool), A, 0.0)

k = ["farm_id", "year", "crop"]
feat = d.groupby(k).agg(n_parcels=("parcel_id", "size"), parcel_area_ha=("parcel_area_ha", "sum"),
                        clay_x_a=("clay_x_a", "sum"), slope_x_a=("slope_x_a", "sum"), org_a=("org_a", "sum"),
                        sandy_a=("sandy_a", "sum"), same_prev_a=("same_prev_a", "sum"), catch_a=("catch_a", "sum"),
                        n_cells=("cell_id", "nunique"))
for src, dst in [("clay_x_a", "clay_pct_wavg"), ("slope_x_a", "slope_pct_wavg"), ("org_a", "organic_share_soilmap"),
                 ("sandy_a", "sandy_share"), ("same_prev_a", "same_crop_prev_year_share"), ("catch_a", "catch_crop_share_parcels")]:
    feat[dst] = feat.pop(src) / feat["parcel_area_ha"]
main_cell = (d.groupby(k + ["cell_id"])["parcel_area_ha"].sum().reset_index()
             .sort_values("parcel_area_ha").drop_duplicates(k, keep="last").set_index(k)["cell_id"])
feat["main_cell_id"] = main_cell
feat = feat.reset_index()

fi_rows = fs_clean.assign(source="fi_farm_survey").drop(
    columns=["decl_area_ha", "org_area", "organic_share_soilmap"])
se_rows = se.assign(source="se_foodhills", n_corrections=0, soil_misreport_flag=False).drop(
    columns=["crop_sv", "irrigated_sv", "decl_area_ha", "org_area", "organic_share_soilmap"])

train = (pd.concat([fi_rows, se_rows], ignore_index=True)
         .merge(fr_clean[["farm_id", "region", "country", "organic_certified", "livestock", "joined_year"]],
                on="farm_id", how="left", validate="many_to_one")
         .merge(feat, on=k, how="left", validate="one_to_one"))

front = ["record_key", "source", "farm_id", "region", "crop", "year", "yield_t_ha", "area_ha", "main_cell_id", "n_cells"]
train = train[front + [c for c in train.columns if c not in front]].sort_values(["region", "crop", "farm_id", "year"])
print(train.shape)
train.groupby(["region", "crop"]).agg(rows=("yield_t_ha", "size"), farms=("farm_id", "nunique"),
                                      years=("year", lambda s: f"{s.min()}-{s.max()}"),
                                      yield_median=("yield_t_ha", "median"), corrected=("n_corrections", "sum"))

(6120, 36)


rows  farms      years  yield_median  corrected
region   crop                                                        
FI_SW    carrot         95     12  2012-2026     50.930000          2
         pea          1043    120  2012-2026      3.270000         44
         potato        136     15  2012-2026     29.210000          5
         spinach       152     16  2012-2026     23.280000         11
         turnip_rape  2011    230  2012-2026      1.460000         85
SE_SKANE pea          2683    300  2018-2026      3.344805          0

## 10. Checks

If any check fails, the notebook stops before writing anything.

In [17]:
checks = {
    "row count = survey + foodhills": len(train) == len(raw["farm_survey"]) + len(raw["foodhills"]),
    "farm x crop x year unique": not train.duplicated(["farm_id", "crop", "year"]).any(),
    "every row has a weather cell": train["main_cell_id"].notna().all(),
    "every row has a yield > 0": (train["yield_t_ha"] > 0).all(),
    "no yield > 5x crop-year median after cleaning":
        (train["yield_t_ha"] / train.groupby(["region", "crop", "year"])["yield_t_ha"].transform("median")).max() < 5,
    "survey area matches parcels (approved R3)":
        ((train["area_ha"] / train["parcel_area_ha"] - 1).abs() < PARAMS["R3_area_tolerance"]).all()
        or log.query("rule_id == 'R3' and status != 'approved'").shape[0] > 0,
    "raw files unchanged during run": all(
        hashlib.sha256((DATA / p.removeprefix("data/")).read_bytes()).hexdigest() == h for p, h in checksums.items()),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)
assert all(checks.values())

PASS row count = survey + foodhills
PASS farm x crop x year unique
PASS every row has a weather cell
PASS every row has a yield > 0
PASS no yield > 5x crop-year median after cleaning
PASS survey area matches parcels (approved R3)
PASS raw files unchanged during run


## 11. Write the outputs

In [18]:
outputs = {
    "corrections_log.csv": log,
    "rule_validation.csv": validation,
    "farm_survey_fi_clean.csv": fs_clean,
    "foodhills_se_clean.csv": se,
    "farm_registry_clean.csv": fr_clean,
    "parcel_features.csv": P,
    "parcel_year_contract.csv": py,
    "c1_training_table.csv": train,
}
for name, df in outputs.items():
    df.to_csv(OUT / name, index=False)
    print(f"{name:28s} {df.shape}")

meta = {"notebook": "c1_clean.ipynb", "run_at": RUN_TS, "package": PKG.name, "params": PARAMS,
        "input_sha256": checksums, "auto_approve_policy": AUTO_APPROVE,
        "corrections": log.groupby(["rule_id", "status"]).size().rename("n").reset_index().to_dict("records"),
        "outputs": {k: list(v.shape) for k, v in outputs.items()}}
_ = (OUT / "run_metadata.json").write_text(json.dumps(meta, indent=2, default=str))

corrections_log.csv          (348, 15)
rule_validation.csv          (5, 11)
farm_survey_fi_clean.csv     (3437, 22)
foodhills_se_clean.csv       (2683, 15)
farm_registry_clean.csv      (680, 12)


parcel_features.csv          (14213, 17)


parcel_year_contract.csv     (48401, 20)
c1_training_table.csv        (6120, 36)


## Points for the method note

- Low yields are kept on purpose. Turnip rape at 0.4–0.6 t/ha looks extreme, but it matches years with pollen beetle or drought. Removing it would make C2 underestimate how often bad years happen.
- The farmers' own soil answers are unreliable by design, so the model uses the soil map. R4 cases go to a person because the soil map is also wrong about one time in eight.
- Yields are reported per farm, but C1 asks for each parcel. Differences between a farm's parcels have to come from soil, clay, slope, previous crop and weather cell (`parcel_year_contract.csv`).
- Not used yet: ERP delivery records, which could cross-check farm deliveries.